In [ ]:
# -------------------
# Cities Coordinates
# -------------------
cities = {
    0:(0,0), 1:(3,4), 2:(6,1),
    3:(7,5), 4:(2,7), 5:(5,3)
}

import math
import random

# -------------------
# Distance Functions
# -------------------
def distance(a, b):
    x1, y1 = cities[a]
    x2, y2 = cities[b]
    return math.sqrt((x1 - x2)**2 + (y1 - y2)**2)

def total_distance(route):
    total = 0
    for i in range(len(route) - 1):
        total += distance(route[i], route[i + 1])
    total += distance(route[-1], route[0])  # return to start
    return total

# -------------------
# Initial Population
# -------------------
def init_population(size):
    population = []
    nodes = list(cities.keys())
    for _ in range(size):
        ind = nodes[:]
        random.shuffle(ind)
        population.append(ind)
    return population

# -------------------
# Tournament Selection
# -------------------
def tournament_selection(pop, k=3):
    selected = random.sample(pop, k)
    return min(selected, key=total_distance)

# -------------------
# Order Crossover (OX)
# -------------------
def cross_over(p1, p2):
    size = len(p1)

    def ox(parent1, parent2):
        child = [None] * size

        # Step 1: Select random segment
        start, end = sorted(random.sample(range(size), 2))

        # Step 2: Copy segment from parent1
        child[start:end+1] = parent1[start:end+1]

        # Step 3: Fill remaining from parent2 in order
        ptr = 0
        for i in range(size):
            if child[i] is None:
                while parent2[ptr] in child:
                    ptr += 1
                child[i] = parent2[ptr]

        return child

    c1 = ox(p1, p2)
    c2 = ox(p2, p1)

    return c1, c2

# -------------------
# Swap Mutation
# -------------------
def swap_mutation(ind, rate=0.1):
    if random.random() < rate:
        i, j = random.sample(range(len(ind)), 2)
        ind[i], ind[j] = ind[j], ind[i]
    return ind

# -------------------
# Genetic Algorithm
# -------------------
def genetic_algo(pop_size, generations=100):
    population = init_population(pop_size)

    best = min(population, key=total_distance)
    best_dist = total_distance(best)

    for gen in range(generations):
        new_pop = []

        while len(new_pop) < pop_size:
            p1 = tournament_selection(population)
            p2 = tournament_selection(population)

            c1, c2 = cross_over(p1, p2)

            c1 = swap_mutation(c1, 0.1)
            c2 = swap_mutation(c2, 0.1)

            new_pop.extend([c1, c2])

        population = new_pop[:pop_size]

        current_best = min(population, key=total_distance)
        curr_distance = total_distance(current_best)

        if curr_distance < best_dist:
            best_dist = curr_distance
            best = current_best

        print(f"Gen {gen+1}: Best Distance = {best_dist:.2f}")

    return best, best_dist

# -------------------
# Run
# -------------------
best_route, best_distance = genetic_algo(pop_size=10, generations=100)

print("\nBest Route:", best_route)
print("Best Distance:", best_distance)